# LQA information monthly transform

In [0]:
import os
import re
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit

app_reg_app_id = os.environ['APP_REG_APP_ID']
TenantId = os.environ['TENANT_ID']

service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")

In [0]:
#ReadStorage = "salandingzonefecradarprd"

#spark.conf.set(f"fs.azure.account.auth.type.{ReadStorage}.dfs.core.windows.net", "OAuth")
#spark.conf.set(f"fs.azure.account.oauth.provider.type.{ReadStorage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
#spark.conf.set(f"fs.azure.account.oauth2.client.id.{ReadStorage}.dfs.core.windows.net", app_reg_app_id)
#spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ReadStorage}.dfs.core.windows.net", service_credential)
#spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ReadStorage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{TenantId}/oauth2/token")

In [0]:
ReadStorage = "salandingzonefecradarprd"
spark.conf.set(f"fs.azure.account.auth.type.{ReadStorage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ReadStorage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ReadStorage}.dfs.core.windows.net", app_reg_app_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ReadStorage}.dfs.core.windows.net", service_credential)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ReadStorage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{TenantId}/oauth2/v2.0/token")

In [0]:
dbutils.fs.ls("abfss://lqa@salandingzonefecradarprd.dfs.core.windows.net/")

In [0]:
prefixes = ["5201-Y-2026-04-30", "6004-Y-2026-04-30", "6005-Y-2026-04-30", "6008-Y-2026-04-30", "6020-Y-2026-04-30", "6023-Y-2026-04-30"]
files = dbutils.fs.ls("abfss://lqa@salandingzonefecradarprd.dfs.core.windows.net/")
selected = [f for f in files if any(f.name.startswith(p) for p in prefixes)]
print(len(selected))

In [0]:
cols = [
    "Reporting_Date",
    "Decision_Date",
    "Account_Manager",
    "WWId_Assessed_client",
    "WWId_Main_borrower",
    "Legal_Name_Main_borrower",
    "CRE_rating__proposed_",
    "Global_Facility_Id",
    "Facility_Description",
    "Local_Facility_ID",
    "FDW_Facility_ID",
    "Source_system",
    "Branch_code",
    "Booking_Entity",
    "Booking_Entity_Code",
    "Originating_Office_Code",
    "Originating_Office",
    "Original_Currency",
    "Outstanding_Amount",
    "Provision_IFRS9",
    "Additional_WriteOff",
    "Total_WriteOff",
    "AssessmentStatus",
    "OnOffBalance",
    "Facility_Status",
    "FacilityType",
    "Has_Approved_Provision_Override_IFRS9"
]


def clean_name(col):
    return re.sub(r'[^A-Za-z0-9_]', '_', col)

def clean_columns(df):
    for col in df.columns:
        df = df.withColumnRenamed(col, clean_name(col))
    return df


In [0]:
#FileInfo(path='abfss://lqa@salandingzonefecradarprd.dfs.core.windows.net/5201-M-2025-06-30.csv', name='5201-M-2025-06-30.csv', size=55497, modificationTime=1751462801000)
dfs = []

for f in selected:
    file_path = f.path
    
    df0 = (
        spark.read
            .option("header", "true")
            .option("delimiter", ";")
            .option("inferSchema", "true")
            .csv(file_path)
            .withColumn("source_file", lit(f.name))
    )
    df = clean_columns(df0)
    #for col in cols:
    #    if col not in df.columns:
    #        df = df.withColumn(col, lit(None))

    dfs.append(df)

big_df = reduce(DataFrame.unionByName, dfs)

display(big_df)

In [0]:
##file_path = 'abfss://lqa@salandingzonefecradarprd.dfs.core.windows.net/5201-M-2026-04-30.csv'

"""df0 = (
        spark.read
            .option("header", "true")
            .option("delimiter", ";")
            .option("inferSchema", "true")
            .csv(file_path)
            .withColumn("source_file", lit("5201"))
    )
display(df0)"""

In [0]:
spark.table("radar.combinedlqa").printSchema()

In [0]:
spark.sql("DROP TABLE IF EXISTS radar.combinedlqa")
big_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("radar.combinedlqa")